# Titanic Dataset: Alternative Data Cleaning and EDA

**AI/ML Task 1**

This is a new, feature-aware workflow. It preserves information from names, cabins, and tickets, then applies a different missing-age method. Put `Titanic-Dataset.csv` beside this notebook before running.

## 1. Import libraries and load data

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', palette='deep')
pd.set_option('display.max_columns', None)
files=['Titanic-Dataset.csv','titanic.csv','Titanic.csv','train.csv']
path=next((Path(x) for x in files if Path(x).exists()), None)
if path is None: raise FileNotFoundError('Place Titanic-Dataset.csv in the same folder as this notebook and run again.')
raw_df=pd.read_csv(path)
print(f'Loaded {path.name}: {raw_df.shape[0]} rows × {raw_df.shape[1]} columns')
raw_df.head()

## 2. Initial data-quality audit

The original data remains in `raw_df`; every cleaning action is performed on a copy.

In [ ]:
audit=pd.DataFrame({'data_type':raw_df.dtypes.astype(str),'missing_values':raw_df.isna().sum(),'missing_percent':(raw_df.isna().mean()*100).round(2),'unique_values':raw_df.nunique(dropna=True)}).sort_values('missing_percent',ascending=False)
display(audit)
print('Exact duplicate rows:',raw_df.duplicated().sum())
print('Duplicate PassengerId values:',raw_df['PassengerId'].duplicated().sum())
display(raw_df.select_dtypes(include=np.number).describe().T)

## 3. Alternative cleaning approach

- Remove only exact duplicate records.
- Extract passenger title, cabin deck, ticket prefix, ticket-group size and family features.
- Impute `Age` with the median of passengers sharing both **Title** and **Pclass**, then use the overall median only as a fallback.
- Fill missing `Embarked` with its mode.
- Keep zero fares and flag them, rather than treating them as errors.

In [ ]:
df=raw_df.drop_duplicates().copy()
df['Title']=df['Name'].str.extract(r',\s*([^.]+)\.',expand=False).str.strip()
title_map={'Mlle':'Miss','Ms':'Miss','Mme':'Mrs','Lady':'Rare','Countess':'Rare','Capt':'Rare','Col':'Rare','Don':'Rare','Dr':'Rare','Major':'Rare','Rev':'Rare','Sir':'Rare','Jonkheer':'Rare','Dona':'Rare'}
df['Title']=df['Title'].replace(title_map)
print('Rows after duplicate check:',len(df))
df['Title'].value_counts()

In [ ]:
df['CabinKnown']=df['Cabin'].notna().astype('int8')
df['Deck']=df['Cabin'].str.extract(r'([A-Za-z])',expand=False).str.upper().fillna('Unknown')
df['TicketPrefix']=(df['Ticket'].astype(str).str.replace(r'\d','',regex=True).str.replace(r'[^A-Za-z]','',regex=True).str.upper().replace('','NUMERIC'))
df['TicketGroupSize']=df.groupby('Ticket')['Ticket'].transform('size')
df['FamilySize']=df['SibSp']+df['Parch']+1
df['IsAlone']=(df['FamilySize']==1).astype('int8')
df['ZeroFare']=(df['Fare']==0).astype('int8')
df[['Deck','TicketPrefix','TicketGroupSize','FamilySize','IsAlone','ZeroFare']].head()

In [ ]:
missing_age=df['Age'].isna().sum()
by_title_class=df.groupby(['Title','Pclass'])['Age'].transform('median')
df['Age']=df['Age'].fillna(by_title_class).fillna(df['Age'].median())
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Survived']=df['Survived'].astype('int8'); df['Pclass']=df['Pclass'].astype('category')
for col in ['Sex','Embarked','Title','Deck','TicketPrefix']: df[col]=df[col].astype('category')
df['AgeGroup']=pd.cut(df['Age'],[0,12,18,35,55,np.inf],labels=['Child (0–12)','Teen (13–18)','Adult (19–35)','Middle (36–55)','Senior (56+)'])
print('Age values imputed:',missing_age)
print('Missing Age after cleaning:',df['Age'].isna().sum())

## 4. Validation after cleaning

In [ ]:
validation=pd.DataFrame({'missing_values':df.isna().sum(),'missing_percent':(df.isna().mean()*100).round(2)}).sort_values('missing_values',ascending=False)
display(validation)
print('Exact duplicates after cleaning:',df.duplicated().sum())
print('Final analytical shape:',df.shape)
assert df['Age'].isna().sum()==0
assert df['Embarked'].isna().sum()==0

## 5. Exploratory data analysis

The charts report associations in the data, not causes of survival.

In [ ]:
overall_rate=df['Survived'].mean()
summary=pd.DataFrame({'Passengers':[len(df)],'Survival rate':[overall_rate],'Median age':[df['Age'].median()],'Median fare':[df['Fare'].median()]})
display(summary.style.format({'Survival rate':'{:.1%}','Median age':'{:.1f}','Median fare':'£{:.2f}'}))

### 5.1 Survival by sex and passenger class

In [ ]:
sex_class=df.groupby(['Pclass','Sex'],observed=False)['Survived'].agg(['mean','count']).reset_index().rename(columns={'mean':'survival_rate','count':'passengers'})
display(sex_class.style.format({'survival_rate':'{:.1%}'}))
fig,ax=plt.subplots(figsize=(9,5)); sns.barplot(data=sex_class,x='Pclass',y='survival_rate',hue='Sex',ax=ax)
ax.set(title='Survival Rate by Sex and Passenger Class',xlabel='Passenger class',ylabel='Survival rate'); ax.yaxis.set_major_formatter(lambda x,_:f'{x:.0%}'); plt.tight_layout(); plt.show()

### 5.2 Survival by age group

In [ ]:
age_results=df.groupby('AgeGroup',observed=False)['Survived'].agg(['mean','count']).reset_index().rename(columns={'mean':'survival_rate','count':'passengers'})
display(age_results.style.format({'survival_rate':'{:.1%}'}))
fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=age_results,x='AgeGroup',y='survival_rate',color='#4C78A8',ax=ax)
ax.set(title='Survival Rate Across Age Groups',xlabel='Age group',ylabel='Survival rate'); ax.tick_params(axis='x',rotation=20); ax.yaxis.set_major_formatter(lambda x,_:f'{x:.0%}'); plt.tight_layout(); plt.show()

### 5.3 Travelling alone versus with family/group

In [ ]:
travel=df.groupby('IsAlone')['Survived'].agg(['mean','count']).reset_index().rename(columns={'mean':'survival_rate','count':'passengers'})
travel['TravelStatus']=travel['IsAlone'].map({0:'With family/group',1:'Alone'})
display(travel[['TravelStatus','survival_rate','passengers']].style.format({'survival_rate':'{:.1%}'}))
fig,ax=plt.subplots(); sns.barplot(data=travel,x='TravelStatus',y='survival_rate',palette=['#72B7B2','#F58518'],ax=ax)
ax.set(title='Survival: Travelling Alone vs With Family',xlabel='',ylabel='Survival rate'); ax.yaxis.set_major_formatter(lambda x,_:f'{x:.0%}'); plt.tight_layout(); plt.show()

### 5.4 Cabin-record availability and survival

In [ ]:
cabin=df.groupby('CabinKnown')['Survived'].agg(['mean','count']).reset_index().rename(columns={'mean':'survival_rate','count':'passengers'})
cabin['CabinRecord']=cabin['CabinKnown'].map({0:'No cabin recorded',1:'Cabin recorded'})
display(cabin[['CabinRecord','survival_rate','passengers']].style.format({'survival_rate':'{:.1%}'}))
fig,ax=plt.subplots(); sns.barplot(data=cabin,x='CabinRecord',y='survival_rate',palette=['#BAB0AC','#54A24B'],ax=ax)
ax.set(title='Survival Rate by Cabin-Record Availability',xlabel='',ylabel='Survival rate'); ax.yaxis.set_major_formatter(lambda x,_:f'{x:.0%}'); plt.tight_layout(); plt.show()

### 5.5 Fare distribution by survival status

In [ ]:
plot_df=df.assign(LogFare=np.log1p(df['Fare']))
fig,ax=plt.subplots(figsize=(9,5)); sns.boxplot(data=plot_df,x='Survived',y='LogFare',hue='Survived',legend=False,palette=['#E45756','#54A24B'],ax=ax)
ax.set(title='Fare Distribution by Survival Status (log scale)',xlabel='Survived (0 = No, 1 = Yes)',ylabel='log(1 + Fare)'); plt.tight_layout(); plt.show()
display(df.groupby('Survived')['Fare'].agg(['median','mean','count']))

## 6. Conclusion

This notebook keeps the original data, uses **Title + Pclass** age imputation, derives useful variables from text and sparse fields, validates the cleaned data and then explores survival through sex/class, age bands, travel context, cabin records and log-transformed fare. The cleaned dataframe, `df`, is ready for future machine-learning work.